1.从环境变量读取DeepSeek apiKey

In [1]:
import os
from dotenv import load_dotenv


load_dotenv()
# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")
print("DeepSeekApiKey:", api_key)

DeepSeekApiKey: sk-0546dbbefe54471ca5e60e67205a2227


2.加载 mfd.md 文件。解析出来主要部分的内容

In [2]:
# from glob import glob
# text_lines = []
# for file_path in glob("./mfd.md", recursive=True):
#     with open(file_path, "r", encoding="utf-8") as file:#encoding="utf-8" 中文使用的GKB会出现编码问题
#         file_text = file.read()
#     text_lines += file_text.split("# ")
# print(text_lines[:1])
# len(text_lines)

import re

with open("./mfd.md", "r", encoding="utf-8") as f:
    file_text = f.read()

# 按 **第xxx条** 拆分，每条法律条文作为一个独立文本块
# 用正则在每个 **第 前面切割，保留分隔符
raw_parts = re.split(r'(?=\*\*第)', file_text)

# 过滤掉空白和非条文内容（如标题行），只保留以 **第 开头的条文
text_lines = [part.strip() for part in raw_parts if part.strip().startswith('**第')]

print(f'共拆分出 {len(text_lines)} 条法律条文')
print(f'示例: {text_lines[0][:80]}...')

共拆分出 387 条法律条文
示例: **第二百零四条** 为了明确物的归属，充分发挥物的效用，保护权利人的合法权益，维护社会经济秩序，制定本编。...


3.准备 LLM 和 Embedding 模型

In [3]:
from openai import OpenAI
from pymilvus import model

#deepseek
deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
)

embedding_model = model.dense.SentenceTransformerEmbeddingFunction(
    model_name='BAAI/bge-small-zh-v1.5',
    device='cpu'
)
# all-MiniLM-L6-v2
# embedding_model = model.dense.SentenceTransformerEmbeddingFunction(
#     model_name='all-MiniLM-L6-v2',
#     device='cpu'
# )

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4.将数据加载到 Milvus

In [4]:
#4_1.创建 MilvusClient
from pymilvus import MilvusClient

# milvus_client = MilvusClient(uri="./milvus_demo.db")
milvus_client = MilvusClient(uri="http://localhost:19530")

collection_name = "mfd_collection"

In [5]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

In [6]:
#定义collection
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=512,
    metric_type="IP",
    consistency_level="Strong",
)

In [7]:
#插入数据
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|██████████| 387/387 [00:00<?, ?it/s]


{'insert_count': 387, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 

In [8]:
# 查看 mfd_collection 中的数据条数和内容
results = milvus_client.query(
    collection_name="mfd_collection",
    filter="id >= 0",
    output_fields=["id", "text"],
    limit=1000
)

print(f"共 {len(results)} 条数据\n")
for r in results[:5]:  # 先看前5条
    print(f"[id={r['id']}] {r['text'][:80]}...")
    print()


共 387 条数据

[id=0] **第二百零四条** 为了明确物的归属，充分发挥物的效用，保护权利人的合法权益，维护社会经济秩序，制定本编。...

[id=1] **第二百零五条** 本编调整因物的归属和利用产生的民事关系。...

[id=2] **第二百零六条** 国家坚持和完善社会主义公有制为主体、多种所有制经济共同发展的基本经济制度。
国家巩固和发展公有制经济，鼓励、支持和引导非公有制经济的发展。...

[id=3] **第二百零七条** 国家、集体、私人的物权和其他权利人的物权受法律平等保护，任何组织或者个人不得侵犯。...

[id=4] **第二百零八条** 不动产权利的设立、变更、转让和消灭，应当依照法律规定登记。动产物权的设立和转让，应当依照法律规定交付。...



5.构建RAG

In [ ]:
# question = "中华人民共和国民法典第二百零五条是什么?"

# search_res = milvus_client.search(
#     collection_name=collection_name,
#     data=embedding_model.encode_queries(
#         [question]
#     ),  # 将问题转换为嵌入向量
#     limit=3,  # 返回前3个结果
#     search_params={"metric_type": "IP", "params": {}},  # 内积距离
#     output_fields=["text"],  # 返回 text 字段
# )

#embedding 模型做的是语义相似度匹配，不是关键词/编号精确匹配
import re
import json

# === 阿拉伯数字 转 中文数字 ===
def num_to_chinese(num):
    """将阿拉伯数字转为中文数字，如 205 -> 二百零五"""
    digits = '零一二三四五六七八九'
    units = ['', '十', '百', '千', '万']
    n = int(num)
    if n == 0:
        return '零'
    parts = []
    s = str(n)
    length = len(s)
    for i, ch in enumerate(s):
        d = int(ch)
        pos = length - 1 - i  # 个位=0, 十位=1, 百位=2...
        if d == 0:
            # 避免连续零和末尾零
            if parts and parts[-1] != '零':
                parts.append('零')
        else:
            parts.append(digits[d] + units[pos])
    # 去掉末尾的零
    result = ''.join(parts).rstrip('零')
    return result

def extract_article_keywords(question):
    """从问题中提取条文编号，支持中文数字和阿拉伯数字，返回所有可能的关键词列表"""
    keywords = []
    # 匹配中文数字条文号: 第二百零五条
    cn_match = re.search(r'第[零一二三四五六七八九十百千万]+条', question)
    if cn_match:
        keywords.append(cn_match.group())
    # 匹配阿拉伯数字条文号: 第205条
    ar_match = re.search(r'第(\d+)条', question)
    if ar_match:
        num = ar_match.group(1)
        cn_num = num_to_chinese(num)
        keywords.append(f'第{cn_num}条')
    return list(set(keywords))  # 去重

# === 搜索逻辑 ===

question = "中华人民共和国民法典第二百零五条是什么?"
# question = "中华人民共和国民法典第205条是什么?"

keywords = extract_article_keywords(question)
print(f'提取到的条文关键词: {keywords}')

exact_matches = []
if keywords:
    for kw in keywords:
        exact_matches += [line for line in text_lines if kw in line]
    exact_matches = list(set(exact_matches))  # 去重

if exact_matches:
    print(f'精确匹配到 {len(exact_matches)} 条结果:')
    for m in exact_matches:
        print(m[:200])
        print()
    retrieved_lines_with_distances = [(m, 1.0) for m in exact_matches[:3]]
else:
    print('未找到精确匹配，回退到向量搜索')
    search_res = milvus_client.search(
        collection_name=collection_name,
        data=embedding_model.encode_queries([question]),
        limit=3,
        search_params={'metric_type': 'IP', 'params': {}},
        output_fields=['text'],
    )
    retrieved_lines_with_distances = [
        (res['entity']['text'], res['distance']) for res in search_res[0]
    ]

#输出搜索结果
print(json.dumps(retrieved_lines_with_distances, indent=4, ensure_ascii=False))

提取到的条文关键词: ['第二百零五条']
精确匹配到 1 条结果:
**第二百零五条** 本编调整因物的归属和利用产生的民事关系。

[
    [
        "**第二百零五条** 本编调整因物的归属和利用产生的民事关系。",
        1.0
    ]
]


In [10]:
# #格式化搜索结果
# import json

# retrieved_lines_with_distances = [
#     (res["entity"]["text"], res["distance"]) for res in search_res[0]
# ]
# print(json.dumps(retrieved_lines_with_distances, indent=4, ensure_ascii=False))

In [11]:
#检索到的文档转换为字符串格式
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

6.将 RAG 返回的关键数据作为上下文 提供给 LLM

In [12]:
#LLM提示词
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。
<context>
{context}
</context>
<question>
{question}
</question>
"""

In [13]:
#生成响应
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

根据您提供的上下文信息，我无法找到《中华人民共和国民法典》第205条的具体内容。您提供的片段中只包含了第四百八十三条、第五百八十九条和第四百八十一条的条文。

如果您需要查询第205条，建议您查阅《中华人民共和国民法典》的官方文本或权威法律数据库以获取准确信息。
